# The 1 July 2026 squall line at ARM's Bankhead National Forest site

**C-SAPR2 dual-polarisation radar + NALMA total lightning**

*Scott Collis and Claude (Anthropic)*

A storm crossed the ARM Bankhead National Forest (BNF) observatory at approximately
19:44 UTC on 1 July 2026. This notebook reproduces an end-to-end analysis of that
event from two independent instrument networks:

| Source | Product | Access |
|---|---|---|
| ARM C-SAPR2 at BNF S3 | `bnfcsapr2cfrS3.a1` — interleaved PPI volumes + 210° RHIs | ARM Live web service |
| ARM sonde at BNF M1 | `bnfsondewnpnM1.b1` — thermodynamic profile | ARM Live web service |
| NASA NALMA | North Alabama Lightning Mapping Array VHF sources | NASA CMR → GHRC DAAC |
| ARM BNF site locations | `BNF.kmz` placemarks | GitHub (ARM-Development) |

**What the analysis shows.** The convective line produced a 60 dBZ core 26 km west of
the M1 supersite. Polarimetry identifies this as heavy rain rather than hail
($Z_{DR}\approx4$ dB, $\rho_{HV}\approx0.97$), but the 40 dBZ echo top reached 13.4 km —
5.5 km above the −20 °C level — indicating a vigorous updraft. Lightning source rate
and 40 dBZ echo-top height rise together through the event (*r* = 0.85), with VHF
emission peaking in the 9–13 km charging layer.

---

## Before you run this

**Credentials are read at runtime and are never stored in this notebook.** Two are needed:

1. **ARM Live** (username + token) — register at [adc.arm.gov/armlive](https://adc.arm.gov/armlive/livedata/home).
   Provide via environment variables `ARM_USER` and `ARM_TOKEN`, or a JSON file (see the setup cell).
2. **NASA Earthdata Login** — register at [urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov).
   Provide via a `~/.netrc` entry. Note the **short username**, not your email address.

**Data volume.** The radar window is ~12 GB (33 volumes) and NALMA is ~38 MB.
All download cells are idempotent — they skip files already present on disk.

## 1. Environment and configuration

`cmweather` supplies the colormaps (`ChaseSpectral` for reflectivity, `balance` for
signed fields, `CM_rhohv` for correlation coefficient).

Two matplotlib settings are load-bearing and explained where they are set.

In [ ]:
import os, json, gzip, pathlib, datetime as dt, urllib.request, urllib.parse, subprocess
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import TwoSlopeNorm, LogNorm
import pandas as pd
import pyart, cmweather, netCDF4
from pyart.core import geographic_to_cartesian_aeqd, cartesian_to_geographic_aeqd

# --- GOTCHA 1 -------------------------------------------------------------
# savefig.bbox='tight' silently mis-crops cartopy GeoAxes: the saved PNG loses
# panels placed at explicit figure rectangles. Force the standard bbox.
mpl.rcParams["savefig.bbox"] = "standard"

# --- GOTCHA 2 -------------------------------------------------------------
# Cartopy downloads coastline/state shapefiles to its package dir on first use.
# If that path is read-only you get PermissionError; redirect to somewhere writable.
CARTOPY_CACHE = pathlib.Path(os.environ.get("CARTOPY_DATA_DIR", pathlib.Path.home()/".cartopy_cache"))
CARTOPY_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["CARTOPY_DATA_DIR"] = str(CARTOPY_CACHE)
import cartopy
cartopy.config["data_dir"] = str(CARTOPY_CACHE)
import cartopy.crs as ccrs, cartopy.feature as cfeature

print("pyart", pyart.__version__, "| cmweather", cmweather.__version__, "| cartopy", cartopy.__version__)

In [ ]:
# ---- Configuration -------------------------------------------------------
DATA_DIR = pathlib.Path(os.environ.get("BNF_DATA_DIR", "~/data/bnf_radar")).expanduser()
STAGE    = DATA_DIR / "csapr2_bnf_20260701_1944UTC"
NALMA_DIR= STAGE / "nalma"
FIG_DIR  = pathlib.Path("BNF_CSAPR2_NALMA_outputs")
for d in (STAGE, NALMA_DIR, FIG_DIR): d.mkdir(parents=True, exist_ok=True)

# --- GOTCHA 3 -------------------------------------------------------------
# BNF C-SAPR2 publishes ONE interleaved datastream (PPI volumes and RHIs in the
# same stream), not separate cfrppi/cfrhsrhi streams as at other ARM sites.
# The event is in 2026 -- there is no BNF C-SAPR2 data for July 2025.
DATASTREAM = "bnfcsapr2cfrS3.a1"
EVENT      = dt.date(2026, 7, 1)
WIN_START  = dt.datetime(2026, 7, 1, 18, 0)
WIN_END    = dt.datetime(2026, 7, 1, 21, 0)
PEAK       = dt.datetime(2026, 7, 1, 19, 44)
RHI_AZ     = 210.0            # fixed RHI azimuth for this scan strategy
T0         = dt.datetime(2026, 7, 1)   # midnight reference for NALMA sec-of-day

print(f"staging to {STAGE}")

## 2. Credentials

Read from the environment, a Claude Science credential store if present, or a local
JSON file — in that order. Nothing is written into the notebook.

To use a file, create it yourself with restrictive permissions:

```bash
printf '{"user":"YOURNAME","token":"YOURTOKEN"}' > ~/.arm_live_creds.json
chmod 600 ~/.arm_live_creds.json
```

In [ ]:
def arm_credentials():
    """Return (user, token) for the ARM Live web service. Never hardcode these."""
    u, t = os.environ.get("ARM_USER"), os.environ.get("ARM_TOKEN")
    if u and t:
        return u, t
    try:                                    # Claude Science credential store
        return host.credentials.get("ARM_USER")["value"], host.credentials.get("ARM_TOKEN")["value"]
    except Exception:
        pass
    for p in (pathlib.Path.home()/".arm_live_creds.json",
              pathlib.Path("~/scratch/.arm_live_creds.json").expanduser()):
        if p.exists():
            d = json.loads(p.read_text())
            return d["user"], d["token"]
    import getpass
    return input("ARM Live username: "), getpass.getpass("ARM Live token: ")

ARM_USER, _ARM_TOKEN = arm_credentials()
print(f"ARM Live user: {ARM_USER}  (token loaded, {len(_ARM_TOKEN)} chars, not displayed)")

## 3. Stage the radar volumes from ARM Live

`query` lists available files for a datastream and date range; `saveData` retrieves one.
The download is threaded and skips anything already on disk.

In [ ]:
def arm_query(datastream, start, end):
    """List files in an ARM datastream between two dates (YYYY-MM-DD)."""
    url = "https://adc.arm.gov/armlive/data/query?" + urllib.parse.urlencode(
        {"user": f"{ARM_USER}:{_ARM_TOKEN}", "ds": datastream,
         "start": str(start), "end": str(end), "wt": "json"})
    with urllib.request.urlopen(url, timeout=60) as r:
        return sorted(json.load(r).get("files", []))

def arm_fetch(fname, dest_dir):
    """Download one ARM file unless already present. Returns (path, cached_flag)."""
    p = pathlib.Path(dest_dir)/fname
    if p.exists() and p.stat().st_size > 0:
        return p, True
    url = "https://adc.arm.gov/armlive/data/saveData?" + urllib.parse.urlencode(
        {"user": f"{ARM_USER}:{_ARM_TOKEN}", "file": fname})
    tmp = p.with_suffix(p.suffix + ".part")
    urllib.request.urlretrieve(url, tmp); tmp.rename(p)
    return p, False

def file_time(fname):
    """Parse the timestamp out of an ARM filename: ds.YYYYMMDD.HHMMSS.nc"""
    parts = fname.split(".")
    return dt.datetime.strptime(parts[2] + parts[3], "%Y%m%d%H%M%S")

available = arm_query(DATASTREAM, EVENT, EVENT + dt.timedelta(days=1))
window = [f for f in available if WIN_START <= file_time(f) <= WIN_END]
print(f"{len(available)} files on {EVENT}; {len(window)} in the {WIN_START:%H:%M}-{WIN_END:%H:%M} UTC window")

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(6) as ex:
    staged = list(ex.map(lambda f: arm_fetch(f, STAGE), window))
n_new = sum(1 for _, cached in staged if not cached)
total_gb = sum(p.stat().st_size for p, _ in staged) / 1e9
print(f"{len(staged)} radar files staged ({n_new} newly downloaded), {total_gb:.1f} GB")

### Separate PPI volumes from RHIs

The interleaved stream mixes two scan types. PPI volumes are 15 sweeps and roughly
40× larger than the single-sweep RHIs, so file size separates them cheaply without
opening every file.

In [ ]:
SIZE_SPLIT = 3e8   # bytes; PPI volumes ~693 MB, RHIs ~18-30 MB

radar_files = sorted(STAGE.glob(f"{DATASTREAM}*.nc"))
ppi_files = [f for f in radar_files if f.stat().st_size >  SIZE_SPLIT]
rhi_files = [f for f in radar_files if f.stat().st_size <= SIZE_SPLIT]

r0 = pyart.io.read(ppi_files[0], include_fields=["reflectivity"], delay_field_loading=True)
RLAT, RLON, RALT = (float(r0.latitude["data"][0]), float(r0.longitude["data"][0]),
                    float(r0.altitude["data"][0]))
NYQUIST = float(r0.get_nyquist_vel(0))
print(f"{len(ppi_files)} PPI volumes, {len(rhi_files)} RHIs")
print(f"C-SAPR2 at {RLAT:.4f} N {RLON:.4f} E, {RALT:.0f} m MSL")
print(f"{r0.nsweeps} sweeps, elevations {np.round(r0.fixed_angle['data'][:8],1)} ...")
print(f"max range {r0.range['data'][-1]/1000:.0f} km | Nyquist {NYQUIST:.1f} m/s")
print(f"{len(r0.fields)} fields available")
del r0

## 4. Site locations from the ARM BNF KMZ

The KMZ holds 41 placemarks, but ~30 are individual instruments packed within ~200 m
at M1. Plotting all of them produces an illegible pile of overlapping labels, so we
keep the spatially distinct facilities and nearby towns.

In [ ]:
import xml.etree.ElementTree as ET, zipfile

# --- GOTCHA 4 -------------------------------------------------------------
# raw.githubusercontent.com returns a 14-byte 404 stub for this path; the
# github.com/.../raw/... URL serves the real file.
KMZ_URL = "https://github.com/ARM-Development/bnf-radar-examples/raw/main/notebooks/locations/BNF.kmz"
kmz = STAGE/"BNF.kmz"
if not (kmz.exists() and zipfile.is_zipfile(kmz)):
    urllib.request.urlretrieve(KMZ_URL, kmz)
assert zipfile.is_zipfile(kmz), "KMZ did not download as a zip archive"

with zipfile.ZipFile(kmz) as z:
    kml = z.read("doc.kml").decode("utf-8", "replace")
NS = {"k": "http://www.opengis.net/kml/2.2"}
placemarks = []
for pm in ET.fromstring(kml).iter("{http://www.opengis.net/kml/2.2}Placemark"):
    name = (pm.findtext("k:name", "", NS) or "").strip()
    coord = (pm.findtext(".//k:Point/k:coordinates", "", NS) or "").strip()
    if coord:
        lon, lat = coord.split(",")[:2]
        placemarks.append((name, float(lat), float(lon)))
print(f"{len(placemarks)} point placemarks in the KMZ")

KEEP  = {"M1", "S3", "S4", "S20", "S30", "S40", "Moulton", "Decatur", "Huntsville"}
TOWNS = {"Moulton", "Decatur", "Huntsville"}
SITES, KIND = {}, {}
for name, lat, lon in placemarks:
    if name in KEEP and name not in SITES:
        SITES[name] = (lat, lon)
        KIND[name] = "town" if name in TOWNS else "arm"
M1_LAT, M1_LON = SITES["M1"]

# The KMZ S3 entry should coincide with the radar's own metadata -- a useful check.
s3_off = np.hypot(*[float(np.atleast_1d(v)[0]) for v in
                    geographic_to_cartesian_aeqd(np.array([SITES["S3"][1]]),
                                                 np.array([SITES["S3"][0]]), RLON, RLAT)])/1000
print(f"KMZ S3 vs radar metadata: {s3_off:.2f} km apart (should be ~0)")
for k, (la, lo) in SITES.items():
    d = np.hypot(*[float(np.atleast_1d(v)[0]) for v in
                   geographic_to_cartesian_aeqd(np.array([lo]), np.array([la]), RLON, RLAT)])/1000
    print(f"  {k:<11} {la:8.4f} {lo:10.4f}  {d:5.1f} km from radar")

## 5. Thermodynamic levels from the pre-storm sounding

The 17:30 UTC M1 sonde launched ~2 h before the peak. The 0 °C and −20 °C heights it
gives are what make the echo-top numbers physically interpretable: the −20 °C level
marks the base of the mixed-phase charging zone.

In [ ]:
sonde_files = arm_query("bnfsondewnpnM1.b1", EVENT, EVENT + dt.timedelta(days=1))
# pick the launch closest to but before the storm peak
pre = [f for f in sonde_files if file_time(f) <= PEAK]
sonde_path, _ = arm_fetch(pre[-1], STAGE)
print("using sounding:", sonde_path.name)

with netCDF4.Dataset(sonde_path) as nc:
    t_dry = np.ma.masked_invalid(nc["tdry"][:])
    alt_m = np.ma.masked_invalid(nc["alt"][:])
    sfc_T, sfc_dp = float(nc["tdry"][0]), float(nc["dp"][0])

def temperature_level(target_C):
    """First height (m MSL) where the profile crosses target_C, linearly interpolated."""
    below = np.where((t_dry[:-1] > target_C) & (t_dry[1:] <= target_C))[0]
    if not len(below):
        return np.nan
    k = below[0]
    frac = (t_dry[k] - target_C) / (t_dry[k] - t_dry[k+1])
    return float(alt_m[k] + frac * (alt_m[k+1] - alt_m[k]))

Z_0C_KM   = (temperature_level(0.0)   - RALT) / 1000    # height above the radar
Z_M20C_KM = (temperature_level(-20.0) - RALT) / 1000
print(f"surface {sfc_T:.1f} C, dewpoint {sfc_dp:.1f} C")
print(f"0 C level    {Z_0C_KM:5.2f} km above radar")
print(f"-20 C level  {Z_M20C_KM:5.2f} km above radar   <- charging zone base")

## 6. Dual-polarisation PPI at the time of closest approach

### Gate filtering

For the *quantitative* polarimetric work below we remove non-meteorological gates:
speckle, low-SNR noise, and depolarised returns. Note `despeckle_field` is a
module-level function that *returns* a `GateFilter` — it is not a `GateFilter` method.

The animation in §9 deliberately skips this filtering to show the raw fields.

In [ ]:
def dualpol_gatefilter(radar, snr_min=8.0, rhohv_min=0.8, z_min=0.0, speckle=25):
    """Conservative gate filter for C-band dual-pol precipitation analysis."""
    gf = pyart.correct.despeckle_field(radar, "reflectivity", size=speckle)
    gf.exclude_masked("reflectivity")
    gf.exclude_below("signal_to_noise_ratio_copolar_h", snr_min)
    gf.exclude_below("copol_correlation_coeff", rhohv_min)
    gf.exclude_below("reflectivity", z_min)
    return gf

PPI_FIELDS = ["reflectivity", "differential_reflectivity", "copol_correlation_coeff",
              "mean_doppler_velocity", "specific_differential_phase",
              "signal_to_noise_ratio_copolar_h"]

# volume closest to the storm's arrival over BNF
peak_ppi = min(ppi_files, key=lambda f: abs(file_time(f.name) - PEAK))
radar = pyart.io.read(peak_ppi, include_fields=PPI_FIELDS)
gf = dualpol_gatefilter(radar)
sweep0 = radar.extract_sweeps([0])
excl = gf.gate_excluded[radar.get_slice(0)]

def sweep_field(name):
    return np.ma.masked_where(excl, sweep0.fields[name]["data"])

PPI_X, PPI_Y, _ = sweep0.get_gate_x_y_z(0)
PPI_X, PPI_Y = PPI_X/1000, PPI_Y/1000
PPI_LON, PPI_LAT = cartesian_to_geographic_aeqd(PPI_X*1000, PPI_Y*1000, RLON, RLAT)
ppi_time = pyart.util.datetime_from_radar(radar)

ZH = sweep_field("reflectivity")
j = np.unravel_index(np.ma.argmax(ZH), ZH.shape)
core_x, core_y = float(PPI_X[j]), float(PPI_Y[j])
core_lon = float(np.atleast_1d(cartesian_to_geographic_aeqd(np.array([core_x*1000]), np.array([core_y*1000]), RLON, RLAT)[0])[0])
core_lat = float(np.atleast_1d(cartesian_to_geographic_aeqd(np.array([core_x*1000]), np.array([core_y*1000]), RLON, RLAT)[1])[0])
m1_xy = [float(np.atleast_1d(v)[0])/1000 for v in
         geographic_to_cartesian_aeqd(np.array([M1_LON]), np.array([M1_LAT]), RLON, RLAT)]
core_to_m1 = np.hypot(core_x - m1_xy[0], core_y - m1_xy[1])

print(f"volume {ppi_time:%H:%M:%S} UTC | gates retained {100*(~excl).sum()/excl.size:.1f}%")
print(f"peak {float(ZH[j]):.1f} dBZ at {np.hypot(core_x,core_y):.1f} km from radar, "
      f"{core_to_m1:.1f} km from M1")

### Is the core hail or heavy rain?

A 60 dBZ core at C band invites a hail interpretation. The polarimetric variables
settle it: hail tumbles and so depolarises the return (low $Z_{DR}$, depressed
$\rho_{HV}$), whereas large oblate raindrops give high $Z_{DR}$ with $\rho_{HV}$
near unity.

In [ ]:
ZDR, RHOHV, KDP = (sweep_field("differential_reflectivity"),
                   sweep_field("copol_correlation_coeff"),
                   sweep_field("specific_differential_phase"))
intense = (ZH >= 50).filled(False)
hail_like = intense & (ZDR < 1.0).filled(False) & (RHOHV < 0.95).filled(False)

print(f"gates with Z >= 50 dBZ: {intense.sum()}")
for label, arr in [("Z_DR (dB)", ZDR), ("rho_HV", RHOHV), ("K_DP (deg/km)", KDP)]:
    v = arr[intense].compressed()
    print(f"  {label:<15} median {np.median(v):6.2f}   p10 {np.percentile(v,10):6.2f}   p90 {np.percentile(v,90):6.2f}")
print(f"\nhail-like gates (Z>=50, Z_DR<1 dB, rho_HV<0.95): {hail_like.sum()}"
      f"  =  {100*hail_like.sum()/max(intense.sum(),1):.1f}% of the intense core")
print("=> high Z_DR with rho_HV ~ 0.97 is heavy rain in large oblate drops, not hail.")

### Figure 1 — dual-polarisation PPI on a map

Two layout notes, both learned the hard way:

* **Do not use `tight_layout` or `constrained_layout` with cartopy `GeoAxes`.** The
  gridliner is evaluated during draw and can raise `LinAlgError: Singular matrix`,
  or silently displace panels. Place axes at explicit figure rectangles instead.
* **Set `x_inline=False, y_inline=False`** on the gridliner, or labels are drawn
  inside the map interior on a Lambert projection.

In [ ]:
PROJ, PC = ccrs.LambertConformal(central_longitude=RLON, central_latitude=RLAT), ccrs.PlateCarree()
GREY = "#888888"
RINGS = (50, 100)
_th = np.linspace(0, 2*np.pi, 361)

PANELS = [
    ("Reflectivity $Z_H$ (dBZ)",                 ZH,    dict(cmap="ChaseSpectral", vmin=-8, vmax=65)),
    ("Differential reflectivity $Z_{DR}$ (dB)",  ZDR,   dict(cmap="balance",
                                                             norm=TwoSlopeNorm(vcenter=0., vmin=-2, vmax=6))),
    ("Co-polar correlation $\\rho_{HV}$",        RHOHV, dict(cmap=cmweather.cm_colorblind.CM_rhohv,
                                                             vmin=0.8, vmax=1.0)),
    ("Doppler velocity (m s$^{-1}$)",            sweep_field("mean_doppler_velocity"),
                                                        dict(cmap="balance",
                                                             norm=TwoSlopeNorm(vcenter=0., vmin=-NYQUIST, vmax=NYQUIST))),
]
# label offsets (dlon, dlat, ha, va) tuned so nothing collides
LABEL_OFF = {"S3": (0.03, 0.05, "left", "bottom"),   "M1": (-0.055, 0.035, "right", "bottom"),
             "S20": (-0.05, 0.02, "right", "bottom"), "S4": (0.05, 0.025, "left", "bottom"),
             "S30": (0.05, -0.01, "left", "center"),  "S40": (0.05, -0.03, "left", "top"),
             "Moulton": (0.045, -0.035, "left", "top"), "Decatur": (0.04, -0.04, "left", "top"),
             "Huntsville": (0.045, -0.015, "left", "center")}
LABEL_TXT = {"S3": "S3 · C-SAPR2", "M1": "M1 supersite"}

FW, FH = 10.6, 10.0
_w = 0.355
_xkm, _ykm = 2.60*111*np.cos(np.deg2rad(RLAT)), 2.12*111
_h = _w*(FW/FH)*(_ykm/_xkm)
_x0, _y0 = [0.078, 0.545], [0.885-_h, 0.885-2*_h-0.055]
EXTENT = [RLON-1.30, RLON+1.30, RLAT-1.10, RLAT+1.02]

fig1 = plt.figure(figsize=(FW, FH))
axes1 = []
for i, (label, data, kw) in enumerate(PANELS):
    row, col = divmod(i, 2)
    ax = fig1.add_axes([_x0[col], _y0[row], _w, _h], projection=PROJ); axes1.append(ax)
    ax.set_extent(EXTENT, crs=PC)
    mesh = ax.pcolormesh(PPI_LON, PPI_LAT, data, transform=PC, shading="nearest",
                         rasterized=True, **kw)
    ax.add_feature(cfeature.STATES.with_scale("10m"), lw=0.5, edgecolor="0.25")
    ax.add_feature(cfeature.RIVERS.with_scale("10m"), lw=0.35, edgecolor="#5a86b8", alpha=0.7)
    ax.add_feature(cfeature.LAKES.with_scale("10m"), lw=0.3, edgecolor="#5a86b8", facecolor="none")
    for rr in RINGS:
        rlon, rlat = cartesian_to_geographic_aeqd(rr*1000*np.sin(_th), rr*1000*np.cos(_th), RLON, RLAT)
        ax.plot(rlon, rlat, transform=PC, color=GREY, lw=0.5, alpha=0.85, zorder=4)
    for name, (la, lo) in SITES.items():
        if KIND[name] == "town":
            ax.plot(lo, la, marker="o", ms=2.2, mfc="0.35", mec="w", mew=0.4, transform=PC, zorder=6)
        elif name == "S3":
            ax.plot(lo, la, marker="^", ms=6, mfc="k", mec="w", mew=0.6, transform=PC, zorder=7)
        else:
            ax.plot(lo, la, marker="s", ms=3.8, mfc="w", mec="k", mew=0.8, transform=PC, zorder=7)
    cax = fig1.add_axes([_x0[col]+_w+0.009, _y0[row], 0.014, _h])
    cb = fig1.colorbar(mesh, cax=cax); cb.set_label(label, fontsize=8); cb.ax.tick_params(labelsize=6.5)
    gl = ax.gridlines(draw_labels=True, lw=0.3, color=GREY, alpha=0.45, ls=":",
                      xlocs=np.arange(-88.5, -85.4, 0.5), ylocs=np.arange(33.5, 35.8, 0.5),
                      x_inline=False, y_inline=False, rotate_labels=False)
    gl.top_labels = gl.right_labels = False
    gl.left_labels, gl.bottom_labels = (col == 0), (row == 1)
    gl.xlabel_style = {"size": 6.5}; gl.ylabel_style = {"size": 6.5}
    fig1.text(_x0[col]-0.052, _y0[row]+_h+0.008, "abcd"[i], fontsize=11, fontweight="bold",
              va="bottom", ha="left")

for name, (la, lo) in SITES.items():
    dlon, dlat, ha, va = LABEL_OFF[name]
    axes1[0].text(lo+dlon, la+dlat, LABEL_TXT.get(name, name), transform=PC, fontsize=6.4,
                  ha=ha, va=va, zorder=9, color="0.3" if KIND[name] == "town" else "k")
axes1[0].text(0.015, 0.985, "▲ C-SAPR2   ■ ARM facility   ● town   rings 50/100 km",
              transform=axes1[0].transAxes, fontsize=6.2, color="0.25", va="top", zorder=9)
fig1.suptitle(
    f"Squall line {core_to_m1:.0f} km west of the BNF M1 supersite: 60 dBZ core with "
    f"$Z_{{DR}}$ ≈ 4 dB and $\\rho_{{HV}}$ ≈ 0.97 — heavy rain, not hail\n"
    f"C-SAPR2 1.5° PPI, {ppi_time:%Y-%m-%d %H:%M:%S} UTC", fontsize=9.5, y=0.965)
fig1.savefig(FIG_DIR/"fig1_ppi_map.png", dpi=200)
print("saved", FIG_DIR/"fig1_ppi_map.png")

## 7. Storm evolution across the PPI volumes

Echo *area* above intensity thresholds measures how the system grows, which is a
different question from how intense its strongest core is. Gate areas scale with
range (the beam fans out), so each gate is weighted by $r\,\Delta\theta\,\Delta r$.

In [ ]:
def sweep_echo_areas(path, thresholds=(30, 40, 50, 55)):
    """Range-weighted echo area (km^2) above each threshold on the lowest sweep."""
    rad = pyart.io.read(path, include_fields=["reflectivity"])
    s0 = rad.extract_sweeps([0])
    z = s0.fields["reflectivity"]["data"]
    x, y, _ = s0.get_gate_x_y_z(0)
    rng_km = np.hypot(x, y)/1000
    dr = float(np.diff(rad.range["data"])[0])/1000
    daz = np.deg2rad(float(np.median(np.abs(np.diff(s0.azimuth["data"][:50])))) or 1.0)
    cell_area = (rng_km*daz)*dr
    valid = ~np.ma.getmaskarray(z)
    out = {"time": pyart.util.datetime_from_radar(rad),
           "max_dbz_volume": float(np.ma.max(rad.fields["reflectivity"]["data"])),
           "max_dbz_sweep0": float(np.ma.max(z))}
    for t in thresholds:
        out[f"area_ge{t}_km2"] = float(cell_area[valid & (z >= t)].sum())
    del rad, s0
    return out

evolution = pd.DataFrame([sweep_echo_areas(f) for f in ppi_files]).sort_values("time")
evolution.to_csv("csapr2_bnf_20260701_storm_metrics.csv", index=False)
growth = evolution["area_ge40_km2"].iloc[-1] / evolution["area_ge40_km2"].iloc[0]
print(f"40 dBZ area grew {growth:.1f}x over the window "
      f"({evolution['area_ge40_km2'].iloc[0]:.0f} -> {evolution['area_ge40_km2'].iloc[-1]:.0f} km2)")
print(f"50 dBZ core area stayed {evolution['area_ge50_km2'].min():.0f}-"
      f"{evolution['area_ge50_km2'].max():.0f} km2 -- footprint expands, cores do not intensify")
evolution[["time","max_dbz_volume","area_ge40_km2","area_ge50_km2"]].head()

## 8. RHI vertical structure and echo tops

The 210° RHI slices straight through the approaching line. The 40 dBZ echo top
relative to the −20 °C level is the diagnostic of interest: precipitation-sized
particles lofted well above −20 °C imply a strong updraft and set up the
non-inductive charging that drives lightning.

In [ ]:
RHI_FIELDS = ["reflectivity", "differential_reflectivity", "mean_doppler_velocity",
              "copol_correlation_coeff", "signal_to_noise_ratio_copolar_h"]

def read_rhi(path, filtered=True):
    """Load one RHI. filtered=True applies the dual-pol gate filter (quantitative use);
    filtered=False masks only sub-noise gates in Z_DR/velocity (display use)."""
    rad = pyart.io.read(path, include_fields=RHI_FIELDS)
    x, y, z = rad.get_gate_x_y_z(0)
    if filtered:
        gf = dualpol_gatefilter(rad)
        keep = lambda n: np.ma.masked_where(gf.gate_excluded, rad.fields[n]["data"])
        Z, D, V = keep("reflectivity"), keep("differential_reflectivity"), keep("mean_doppler_velocity")
    else:
        snr = rad.fields["signal_to_noise_ratio_copolar_h"]["data"]
        no_signal = np.ma.filled(snr, -999) < 0.0
        Z = rad.fields["reflectivity"]["data"]      # fully raw
        D = np.ma.masked_where(no_signal, rad.fields["differential_reflectivity"]["data"])
        V = np.ma.masked_where(no_signal, rad.fields["mean_doppler_velocity"]["data"])
    out = dict(rng=np.hypot(x, y)/1000, ht=z/1000, Z=Z, D=D, V=V,
               az=float(rad.fixed_angle["data"][0]),
               time=pyart.util.datetime_from_radar(rad))
    del rad
    return out

def echo_top(scan, threshold=40.0):
    """Highest gate (km) at or above a reflectivity threshold; NaN if none present."""
    m = (scan["Z"] >= threshold).filled(False) if np.ma.isMaskedArray(scan["Z"]) else (scan["Z"] >= threshold)
    return float(scan["ht"][m].max()) if m.any() else np.nan

rhi_filtered = [read_rhi(f, filtered=True) for f in rhi_files]

# --- GOTCHA 5 -------------------------------------------------------------
# Py-ART may return cftime.DatetimeGregorian, which matplotlib cannot plot.
# Convert to stdlib datetime before any time-axis plotting.
def to_datetime(t):
    if isinstance(t, dt.datetime):
        return t
    return dt.datetime(t.year, t.month, t.day, t.hour, t.minute,
                       int(t.second), int((t.second - int(t.second))*1e6))

rhi_times = np.array([to_datetime(s["time"]) for s in rhi_filtered])
rhi_top40 = np.array([echo_top(s, 40.0) for s in rhi_filtered])
m1_range = np.hypot(*m1_xy)

for t, top in zip(rhi_times, rhi_top40):
    flag = "  <- above -20 C" if top > Z_M20C_KM else ""
    print(f"  {t:%H:%M:%S}  40 dBZ top {top:5.2f} km" if not np.isnan(top)
          else f"  {t:%H:%M:%S}  no 40 dBZ echo on this azimuth", flag)
print(f"\nM1 lies {m1_range:.1f} km along the {RHI_AZ:.0f}° azimuth")

## 9. NALMA total lightning

### Access path

NALMA lives at NASA's GHRC DAAC, not in the ARM archive. Getting the bytes takes three
hops, and the middle one is where people get stuck:

1. **CMR** (`cmr.earthdata.nasa.gov`) — search for granules covering the event window.
2. **GHRC** (`data.ghrc.earthdata.nasa.gov`) — the URL in each granule record. Returns
   a 302 to Earthdata Login.
3. **CloudFront** (`*.cloudfront.net`) — after authentication, GHRC issues a *signed
   redirect* to a CDN host. If you are behind a network allowlist, this third host must
   also be permitted; authentication succeeding is not sufficient.

Authentication uses `.netrc` with your **short EDL username** — not your email address.
Supplying the email produces a 401 that looks exactly like a wrong password.

Create the `.netrc` yourself (outside this notebook) so no password is ever stored here:

```bash
printf 'machine urs.earthdata.nasa.gov login YOURNAME password YOURPASSWORD\n' >> ~/.netrc
chmod 600 ~/.netrc
```

One-time step in your EDL profile: **Applications → Authorized Apps → approve GHRC DAAC**.
Without it every download 401s regardless of credentials.

In [ ]:
NETRC = pathlib.Path(os.environ.get("NETRC_PATH", pathlib.Path.home()/".netrc")).expanduser()

def cmr_granules(short_name, t_start, t_end, provider="GHRC_DAAC"):
    """Find granule download URLs for a CMR collection over a time range."""
    c_url = "https://cmr.earthdata.nasa.gov/search/collections.json?" + urllib.parse.urlencode(
        {"short_name": short_name, "provider": provider})
    with urllib.request.urlopen(c_url, timeout=60) as r:
        entries = json.load(r)["feed"]["entry"]
    cid = entries[0]["id"]
    g_url = "https://cmr.earthdata.nasa.gov/search/granules.json?" + urllib.parse.urlencode(
        {"collection_concept_id": cid, "temporal": f"{t_start:%Y-%m-%dT%H:%M:%SZ},{t_end:%Y-%m-%dT%H:%M:%SZ}",
         "page_size": 200, "sort_key": "start_date"})
    with urllib.request.urlopen(g_url, timeout=60) as r:
        granules = json.load(r)["feed"]["entry"]
    out = []
    for g in granules:
        links = [l["href"] for l in g.get("links", []) if l.get("rel", "").endswith("/data#")]
        if links:
            out.append((g["title"], links[0]))
    return cid, out

collection_id, granules = cmr_granules("nalma", WIN_START, WIN_END)
print(f"collection {collection_id}: {len(granules)} granules in the window")
print("  e.g.", granules[0][0])

In [ ]:
def earthdata_fetch(url, dest):
    """Download one Earthdata-protected file via curl, following the signed CDN redirect.

    curl handles the OAuth redirect chain with --location-trusted plus a cookie jar;
    urllib does not follow it cleanly. Requires a valid ~/.netrc entry.
    """
    dest = pathlib.Path(dest)
    if dest.exists() and dest.stat().st_size > 1000:
        return dest, True
    cookies = dest.parent/".edl_cookies"
    cmd = ["curl", "-sSL", "--netrc-file", str(NETRC), "--location-trusted",
           "-b", str(cookies), "-c", str(cookies),
           "-o", str(dest), "--max-time", "300", url]
    subprocess.run(cmd, check=True)
    # A 401 body is a few dozen bytes of text, not a gzip stream -- catch it early.
    if dest.stat().st_size < 1000:
        msg = dest.read_text(errors="replace")[:120]
        dest.unlink()
        raise RuntimeError(f"download failed ({msg!r}). Check the short EDL username "
                           f"in {NETRC}, and that GHRC DAAC is an authorized app.")
    return dest, False

nalma_paths = []
for title, url in granules:
    p, cached = earthdata_fetch(url, NALMA_DIR/title)
    nalma_paths.append(p)
print(f"{len(nalma_paths)} NALMA granules staged, "
      f"{sum(p.stat().st_size for p in nalma_paths)/1e6:.0f} MB")

### Parse and quality-control the VHF sources

Each file is an ASCII header followed by one line per VHF source: seconds-of-day,
latitude, longitude, altitude, reduced $\chi^2$, power, and a station mask.

The solver accepts solutions up to $\chi^2 \le 5$, but standard practice for
scientific use is much tighter. We require $\chi^2 \le 1$ and at least 7 contributing
stations, which retains ~13% of raw sources.

In [ ]:
def read_nalma(path):
    """Parse a NALMA analysed-data file into an (N,7) array.

    Columns: sec_of_day, lat, lon, alt_m, reduced_chi2, power_dBW, n_stations
    """
    rows, in_data = [], False
    with gzip.open(path, "rt", errors="replace") as f:
        for line in f:
            if not in_data:
                in_data = line.startswith("*** data ***")
                continue
            p = line.split()
            if len(p) < 7:
                continue
            rows.append((float(p[0]), float(p[1]), float(p[2]), float(p[3]),
                         float(p[4]), float(p[5]), bin(int(p[6], 16)).count("1")))
    return np.array(rows)

CHI2_MAX, NSTA_MIN, ALT_MAX_KM = 1.0, 7, 20.0

raw = np.vstack([read_nalma(p) for p in nalma_paths])
src_alt_km = raw[:, 3]/1000
quality = (raw[:, 4] <= CHI2_MAX) & (raw[:, 6] >= NSTA_MIN) & (src_alt_km <= ALT_MAX_KM)
src = raw[quality]
src_sec, src_alt = src[:, 0], src[:, 3]/1000

print(f"{raw.shape[0]:,} raw sources -> {src.shape[0]:,} after QC "
      f"(chi2<={CHI2_MAX}, >={NSTA_MIN} stations, alt<={ALT_MAX_KM:.0f} km) "
      f"= {100*src.shape[0]/raw.shape[0]:.1f}%")

# NALMA network centre, from the file header -- used to judge retrieval quality by range
NALMA_CENTRE = (34.72461, -86.64533)
_d = [float(np.atleast_1d(v)[0]) for v in
      geographic_to_cartesian_aeqd(np.array([NALMA_CENTRE[1]]), np.array([NALMA_CENTRE[0]]),
                                   M1_LON, M1_LAT)]
print(f"NALMA centre is {np.hypot(*_d)/1000:.0f} km from BNF M1 -- "
      f"source ALTITUDES carry ~km-scale error at this range; flash RATES are robust")

In [ ]:
# geometry: source positions relative to the radar, and to the RHI plane
src_x, src_y = geographic_to_cartesian_aeqd(src[:, 2], src[:, 1], RLON, RLAT)
src_x, src_y = np.asarray(src_x).ravel()/1000, np.asarray(src_y).ravel()/1000

beam = np.array([np.sin(np.deg2rad(RHI_AZ)), np.cos(np.deg2rad(RHI_AZ))])
src_along =  src_x*beam[0] + src_y*beam[1]      # distance along the RHI azimuth
src_cross = -src_x*beam[1] + src_y*beam[0]      # perpendicular distance from the plane

# storm region: within 40 km of M1, for the rate time series
m1_dx, m1_dy = geographic_to_cartesian_aeqd(src[:, 2], src[:, 1], M1_LON, M1_LAT)
src_r_m1 = np.hypot(np.asarray(m1_dx).ravel(), np.asarray(m1_dy).ravel())/1000
STORM_REGION = src_r_m1 <= 40.0

# RHI plane selection: near the slice and in front of the radar
PLANE_HALF_WIDTH = 10.0
IN_PLANE = (np.abs(src_cross) <= PLANE_HALF_WIDTH) & (src_along > 0) & (src_along <= 110)

print(f"{STORM_REGION.sum():,} sources within 40 km of M1")
print(f"{IN_PLANE.sum():,} sources within ±{PLANE_HALF_WIDTH:.0f} km of the {RHI_AZ:.0f}° plane")
print(f"source altitude: median {np.median(src_alt[STORM_REGION]):.1f} km, "
      f"IQR {np.percentile(src_alt[STORM_REGION],25):.1f}-{np.percentile(src_alt[STORM_REGION],75):.1f} km")

In [ ]:
# 2-minute source-rate time series over the storm region
BIN_S = 120.0
bin_edges = np.arange(WIN_START.hour*3600, WIN_END.hour*3600 + BIN_S, BIN_S)
rate_counts, _ = np.histogram(src_sec[STORM_REGION], bins=bin_edges)
rate_times = np.array([T0 + dt.timedelta(seconds=float(e)) for e in bin_edges[:-1] + BIN_S/2])
_bin_of = np.digitize(src_sec[STORM_REGION], bin_edges) - 1
_alt_storm = src_alt[STORM_REGION]
rate_med_alt = np.array([np.median(_alt_storm[_bin_of == k]) if (_bin_of == k).sum() >= 20 else np.nan
                         for k in range(len(bin_edges)-1)])

peak_i = int(rate_counts.argmax())
print(f"peak {rate_counts[peak_i]:,} sources/2 min at {rate_times[peak_i]:%H:%M} UTC "
      f"(storm reached BNF ~{PEAK:%H:%M})")
print(f"total in storm region: {rate_counts.sum():,}")

pd.DataFrame({"time_utc": [t.isoformat() for t in rate_times],
              "vhf_sources_per_2min": rate_counts.astype(int),
              "median_source_alt_km": np.round(rate_med_alt, 2)}
             ).to_csv("nalma_bnf_20260701_source_rate.csv", index=False)

### Do lightning and updraft depth covary?

Both series are sampled irregularly (radar every ~8 min, lightning binned every 2 min),
so the lightning rate is interpolated onto the radar scan times before correlating.

In [ ]:
valid = ~np.isnan(rhi_top40)
rate_at_scan = np.interp([(t - T0).total_seconds() for t in rhi_times],
                         bin_edges[:-1] + BIN_S/2, rate_counts.astype(float))
r_zero_lag = np.corrcoef(rhi_top40[valid], rate_at_scan[valid])[0, 1]
print(f"Pearson r (40 dBZ top vs VHF rate) = {r_zero_lag:.2f}  (n = {valid.sum()})")

print("\nlagged correlations (positive lag = lightning shifted later):")
for lag_min in (-16, -8, 0, 8, 16):
    shifted = np.interp([(t - T0).total_seconds() + lag_min*60 for t in rhi_times],
                        bin_edges[:-1] + BIN_S/2, rate_counts.astype(float))
    print(f"  {lag_min:+3d} min: r = {np.corrcoef(rhi_top40[valid], shifted[valid])[0,1]:.2f}")
print("\nThe lag curve is flat: with n=13 radar scans this CANNOT resolve whether")
print("lightning leads or lags echo-top growth. Reported as covariation only.")

pd.DataFrame({"time_utc": [t.isoformat() for t in rhi_times],
              "rhi_40dbz_top_km": np.round(rhi_top40, 2),
              "vhf_rate_interp_per_2min": np.round(rate_at_scan, 1)}
             ).to_csv("nalma_csapr2_echotop_pairs.csv", index=False)

### Figure 2 — lightning distribution and its relationship to updraft depth

In [ ]:
fig2 = plt.figure(figsize=(9.6, 7.6))
ax_map = fig2.add_axes([0.075, 0.615, 0.375, 0.335])
ax_hist = fig2.add_axes([0.620, 0.615, 0.305, 0.335])
ax_ts = fig2.add_axes([0.075, 0.085, 0.85, 0.395])

# (a) plan-view source density
sel = STORM_REGION
Hd, xe, ye = np.histogram2d(src_x[sel], src_y[sel],
                           bins=[np.arange(-80, 20, 2), np.arange(-70, 30, 2)])
mesh = ax_map.pcolormesh(xe, ye, np.ma.masked_equal(Hd, 0).T, cmap="plasmidis",
                         norm=LogNorm(vmin=1, vmax=max(Hd.max(), 10)))
for rr in (25, 50, 75):
    ax_map.plot(rr*np.cos(_th), rr*np.sin(_th), color=GREY, lw=0.4, alpha=0.7, zorder=4)
ax_map.plot(0, 0, marker="^", ms=6, mfc="k", mec="w", mew=0.6, zorder=6)
ax_map.plot(*m1_xy, marker="s", ms=5, mfc="w", mec="k", mew=0.9, zorder=6)
ax_map.text(3, 3, "C-SAPR2", fontsize=6.3, zorder=7)
ax_map.text(m1_xy[0]+3, m1_xy[1]-3, "BNF M1", fontsize=6.3, va="top", zorder=7)
ax_map.set_aspect("equal"); ax_map.set_xlim(-80, 25); ax_map.set_ylim(-75, 25)
ax_map.set_xlabel("East of radar (km)"); ax_map.set_ylabel("North of radar (km)")
ax_map.text(0.02, 0.02, "rings 25 km", transform=ax_map.transAxes, fontsize=6.0, color=GREY)
cb = fig2.colorbar(mesh, cax=fig2.add_axes([0.462, 0.615, 0.013, 0.335]))
cb.set_label("VHF sources per 4 km$^2$", fontsize=7.5); cb.ax.tick_params(labelsize=6.3)
ax_map.set_title("Lightning tracks the line southwest of BNF", fontsize=8.8, loc="left")

# (b) altitude distribution against the charging zone
ax_hist.hist(src_alt[sel], bins=np.arange(0, 20.5, 0.5), orientation="horizontal",
             color="#4a6fa5", edgecolor="none")
_xl = ax_hist.get_xlim()
for zl, lb in [(Z_0C_KM, "0 °C"), (Z_M20C_KM, "−20 °C")]:
    ax_hist.axhline(zl, color="0.15", lw=0.7, ls="-.", zorder=5)
    ax_hist.text(_xl[1]*0.96, zl+0.3, lb, fontsize=6.3, ha="right", color="0.15")
ax_hist.set_ylim(0, 20); ax_hist.set_xlim(_xl)
ax_hist.set_ylabel("Source altitude (km MSL)"); ax_hist.set_xlabel("VHF sources")
ax_hist.set_title("Emission peaks near −20 °C", fontsize=8.8, loc="left")
ax_hist.text(0.96, 0.045, f"median {np.median(src_alt[sel]):.1f} km",
             transform=ax_hist.transAxes, fontsize=6.6, ha="right")

# (c) rate against echo top -- twin axes, colour-coded to each series
ax_ts.bar(rate_times, rate_counts, width=dt.timedelta(seconds=BIN_S*0.9),
          color="#c8a2c8", edgecolor="none", zorder=2)
ax_ts.set_ylabel("VHF sources per 2 min", color="#7b4a7b")
ax_ts.tick_params(axis="y", labelcolor="#7b4a7b")
ax_ts.set_xlabel(f"{EVENT:%Y-%m-%d} time (UTC)")
ax_ts.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax_ts.set_xlim(WIN_START, WIN_END); ax_ts.set_ylim(0, rate_counts.max()*1.07)
ax_top = ax_ts.twinx()
# plot only valid points so the line does not bridge scans with no 40 dBZ echo
ax_top.plot(rhi_times[valid], rhi_top40[valid], "-", lw=1.4, color="#b2182b", zorder=4)
ax_top.plot(rhi_times[valid], rhi_top40[valid], "o", ms=3.6, color="#b2182b", zorder=5)
ax_top.set_ylabel("40 dBZ echo top (km)", color="#b2182b")
ax_top.tick_params(axis="y", labelcolor="#b2182b"); ax_top.set_ylim(0, 17)
ax_top.axhline(Z_M20C_KM, color="0.15", lw=0.6, ls="-.", zorder=3)
ax_top.text(WIN_END - dt.timedelta(minutes=4), Z_M20C_KM+0.35, "−20 °C",
            fontsize=6.2, color="0.15", ha="right")
ax_ts.text(0.005, 0.965,
           f"VHF source rate (bars, left)   ·   40 dBZ echo top on the {RHI_AZ:.0f}° RHI "
           f"(line, right)   ·   r = {r_zero_lag:.2f} (n = {valid.sum()})",
           transform=ax_ts.transAxes, fontsize=7, va="top")
ax_ts.text(0.005, 0.905, "line breaks = scans with no 40 dBZ echo on that azimuth",
           transform=ax_ts.transAxes, fontsize=6.2, va="top", color=GREY)
ax_ts.set_title("Lightning rate and updraft depth rise together through the event",
                fontsize=9, loc="left")
for letter, (lx, ly) in zip("abc", [(0.016, 0.965), (0.560, 0.965), (0.016, 0.505)]):
    fig2.text(lx, ly, letter, fontsize=11, fontweight="bold", va="top")
fig2.text(0.5, 0.995, f"NALMA total lightning over the BNF squall line, "
          f"{EVENT:%Y-%m-%d} {WIN_START:%H:%M}–{WIN_END:%H:%M} UTC",
          fontsize=10, ha="center", va="top")
fig2.savefig(FIG_DIR/"fig2_lightning.png", dpi=200)
print("saved", FIG_DIR/"fig2_lightning.png")

## 10. Animation: raw RHI fields with lightning overlaid

This is the payoff figure. Three points of method:

* **No gate filtering.** $Z_H$ is fully raw so the weak anvil, shallow low-level
  returns and clutter are all visible. Only $Z_{DR}$ and velocity get a sub-noise mask
  (SNR < 0 dB), because those fields are pure noise without signal and would otherwise
  swamp the panels.
* **Velocity is aliased.** Folding is visible aloft after ~19:20 as abrupt red↔blue
  transitions. Dealias before any quantitative velocity work.
* **The lightning overlay is a projection.** Sources within ±2 min of the scan and
  ±10 km of the plane are plotted at their along-plane range and true altitude. Some
  belong to convection adjacent to the slice rather than in it — unavoidable when
  collapsing a 3-D point cloud onto a 2-D cut, which is why the tolerance is stated
  in-panel.

In [ ]:
import matplotlib.animation as manim

rhi_raw = [read_rhi(f, filtered=False) for f in rhi_files]
LIGHTNING_WINDOW_S = 120.0

FW_A, FH_A = 9.2, 8.8
_H, _GAP, _X, _W, _TOP = 0.225, 0.037, 0.082, 0.775, 0.895
fig3 = plt.figure(figsize=(FW_A, FH_A))
ax_z = fig3.add_axes([_X, _TOP-_H,            _W, _H])
ax_d = fig3.add_axes([_X, _TOP-2*_H-_GAP,     _W, _H])
ax_v = fig3.add_axes([_X, _TOP-3*_H-2*_GAP,   _W, _H])
ax_r = fig3.add_axes([_X, 0.062,              _W, 0.105])
cax_z = fig3.add_axes([_X+_W+0.010, _TOP-_H,          0.014, _H])
cax_d = fig3.add_axes([_X+_W+0.010, _TOP-2*_H-_GAP,   0.014, _H])
cax_v = fig3.add_axes([_X+_W+0.010, _TOP-3*_H-2*_GAP, 0.014, _H])

ANIM_PANELS = [
    (ax_z, cax_z, "Z", "Reflectivity $Z_H$ (dBZ)", dict(cmap="ChaseSpectral", vmin=-32, vmax=65)),
    (ax_d, cax_d, "D", "$Z_{DR}$ (dB)",            dict(cmap="balance",
                                                        norm=TwoSlopeNorm(vcenter=0., vmin=-4, vmax=6))),
    (ax_v, cax_v, "V", "Radial velocity (m s$^{-1}$)", dict(cmap="balance",
                                                        norm=TwoSlopeNorm(vcenter=0., vmin=-NYQUIST, vmax=NYQUIST))),
]
meshes, scatters = {}, {}
first = rhi_raw[0]
for ax, cax, key, label, kw in ANIM_PANELS:
    meshes[key] = ax.pcolormesh(first["rng"], first["ht"], first[key],
                                shading="nearest", rasterized=True, **kw)
    cb = fig3.colorbar(meshes[key], cax=cax)
    cb.set_label(label, fontsize=7.6); cb.ax.tick_params(labelsize=6.3)
    ax.set_ylim(0, 17); ax.set_xlim(0, 110); ax.set_ylabel("Height (km)")
    ax.axvline(m1_range, color="k", lw=0.8, ls="--", zorder=6)
    for zl in (Z_0C_KM, Z_M20C_KM):
        ax.axhline(zl, color="0.15", lw=0.6, ls="-.", zorder=6)
    # black open circles read against every colormap used here
    scatters[key] = ax.scatter([], [], s=4.5, facecolors="none", edgecolors="k",
                               linewidths=0.32, alpha=0.60, zorder=8)
    # temperature labels go right in the velocity panel to clear the data
    xpos, ha = (106.0, "right") if ax is ax_v else (1.5, "left")
    for zl, lb in [(Z_0C_KM, "0 °C"), (Z_M20C_KM, "−20 °C")]:
        ax.text(xpos, zl+0.35, lb, fontsize=6.0, color="0.12", ha=ha, zorder=9,
                bbox=dict(fc="w", ec="none", alpha=0.75, pad=0.4))

ax_z.set_xticklabels([]); ax_d.set_xticklabels([])
ax_v.set_xlabel(f"Range along {RHI_AZ:.0f}° azimuth from C-SAPR2 (km)")
ax_z.text(m1_range-1.2, 16.3, "M1", fontsize=6.4, ha="right", va="top", zorder=9,
          bbox=dict(fc="w", ec="none", alpha=0.75, pad=0.4))
ax_z.text(0.995, 0.955,
          f"○ NALMA VHF sources, ±{LIGHTNING_WINDOW_S/60:.0f} min and "
          f"±{PLANE_HALF_WIDTH:.0f} km of the RHI plane",
          transform=ax_z.transAxes, fontsize=6.3, ha="right", va="top", zorder=9,
          bbox=dict(fc="w", ec="none", alpha=0.85, pad=0.8))
title = ax_z.set_title("", fontsize=9.5, loc="left")

ax_r.bar(rate_times, rate_counts, width=dt.timedelta(seconds=BIN_S*0.9),
         color="#c8a2c8", edgecolor="none", zorder=2)
ax_r.set_xlim(WIN_START, WIN_END); ax_r.set_ylim(0, rate_counts.max()*1.07)
ax_r.set_ylabel("VHF src\nper 2 min", fontsize=7.2)
ax_r.set_xlabel(f"{EVENT:%Y-%m-%d} time (UTC)")
ax_r.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M")); ax_r.tick_params(labelsize=6.8)
ax_r.text(0.995, 0.90, "storm-region source rate; red line = current frame",
          transform=ax_r.transAxes, fontsize=6.0, ha="right", va="top", color=GREY)
cursor = ax_r.axvline(to_datetime(rhi_raw[0]["time"]), color="#b2182b", lw=1.3, zorder=5)

def draw_frame(i):
    scan = rhi_raw[i]
    for _, _, key, _, _ in ANIM_PANELS:
        meshes[key].set_array(scan[key].ravel())
    t = to_datetime(scan["time"])
    near = IN_PLANE & (np.abs(src_sec - (t - T0).total_seconds()) <= LIGHTNING_WINDOW_S)
    pts = np.column_stack([src_along[near], src_alt[near]])
    for key in scatters:
        scatters[key].set_offsets(pts if len(pts) else np.empty((0, 2)))
    cursor.set_xdata([t, t])
    title.set_text(f"{t:%H:%M:%S} UTC   ·   {int(near.sum())} VHF sources in plane   "
                   f"·   max $Z_H$ {float(np.ma.max(scan['Z'])):.1f} dBZ")
    return []

draw_frame(len(rhi_raw)//2)
print(f"{len(rhi_raw)} frames ready")

In [ ]:
anim = manim.FuncAnimation(fig3, draw_frame, frames=len(rhi_raw), blit=False)
mp4 = FIG_DIR/"rhi_lightning_animation.mp4"
anim.save(mp4, writer=manim.FFMpegWriter(fps=2, bitrate=4200, codec="libx264",
          extra_args=["-pix_fmt", "yuv420p", "-crf", "20"]), dpi=150)
print(f"saved {mp4} ({mp4.stat().st_size/1e6:.1f} MB)")

from IPython.display import Video
Video(str(mp4), embed=False, width=780)

## 11. Summary

**Storm structure.** A 60 dBZ core passed 26 km west of the M1 supersite at 19:45 UTC.
Polarimetry identifies it as heavy rain in large oblate drops — median $Z_{DR}$ of
4.1 dB and $\rho_{HV}$ of 0.97 in the Z ≥ 50 dBZ gates, with only ~1% of those gates
meeting a hail signature.

**Updraft depth.** The 40 dBZ echo top reached 13.4 km at 19:44 — 5.5 km above the
−20 °C level from the pre-storm sounding. The deepening was abrupt: tops went from
5.7 km at 19:04 to 12.8 km at 19:20, crossing the charging-zone base within one
8-minute scan interval.

**System growth.** The 40 dBZ echo area more than quadrupled over the window while
the 50 dBZ core area stayed near-constant — the system expanded its heavy-rain
footprint rather than intensifying its cores.

**Lightning.** VHF source rate peaked at ~3,000 per 2 min at 19:43, coincident with
the line crossing BNF, and correlates with the 40 dBZ echo top at *r* = 0.85. Source
altitudes cluster at a median 11.3 km, straddling and mostly above the −20 °C level —
the non-inductive charging zone. The same updraft lofting precipitation above −20 °C
is driving the electrification.

### Caveats

| Issue | Consequence |
|---|---|
| No attenuation correction | C-band $Z_H$ and $Z_{DR}$ are biased low behind the 60 dBZ core; visible as the negative $Z_{DR}$ streak beyond ~40 km. Correct before quantitative rain rates. |
| Velocity is aliased | Nyquist is ±16.3 m s⁻¹; folding aloft after 19:20. Dealias before kinematic analysis. |
| RHI is a fixed azimuth | Apparent intensity changes partly reflect the storm translating across the 210° plane. PPI areas are the better growth measure. |
| NALMA range | BNF is 76 km from the network centre, storm sources a median 93 km. Altitudes carry ~km error; rates are robust. |
| n = 13 for the correlation | Sufficient for covariation, not for resolving lightning lead/lag. Use full PPI cadence for that. |

### Natural extensions

1. Attenuation-correct $Z_H$/$Z_{DR}$ and derive rain rate from specific attenuation.
2. Dealias velocity, then grid volumes for a 3-D kinematic retrieval.
3. Cluster VHF sources into flashes so the overlay shows initiation points.
4. Grid to CAPPI and compute VIL / updraft volume above −20 °C.